In [35]:
import json
from pathlib import Path

# 1. ระบุโฟลเดอร์ที่เก็บไฟล์ JSON ทั้ง 50+ ไฟล์ไว้
folder_path = Path('./JSON')

combined_data = []

# 2. ใช้ .glob เพื่อกวาดหาไฟล์ .json ทั้งหมดในโฟลเดอร์นั้น
# ไฟล์จะเยอะแค่ไหน 50 หรือ 500 ไฟล์ มันก็จะวนลูปอ่านจนครบ
for file_path in folder_path.glob('*.json'):
    with open(file_path, 'r', encoding='utf-8') as f:
        try:
            data = json.load(f)
            # ถ้าข้างในเป็น List อยู่แล้วให้ extend แต่ถ้าเป็น Dict ให้ append
            if isinstance(data, list):
                combined_data.extend(data)
            else:
                combined_data.append(data)
        except Exception as e:
            print(f"อ่านไฟล์ {file_path.name} ไม่ได้: {e}")

print(f"ดึงข้อมูลเสร็จสิ้น! ได้ข้อมูลรวมทั้งหมด {len(combined_data)} รายการ")

ดึงข้อมูลเสร็จสิ้น! ได้ข้อมูลรวมทั้งหมด 3556 รายการ


## 📖 วิธีการใช้ Notebook นี้

### ⚠️ **สำคัญ: ต้องรัน Cells ตามลำดับนี้เท่านั้น!**

#### **ขั้นที่ 1️⃣ - โหลดและเตรียมข้อมูล:**
- **Cell 1**: โหลดไฟล์ JSON ทั้งหมดจากโฟลเดอร์ `./JSON/`
- **Cell 2**: บันทึก combined data เป็นไฟล์ `all_files_combined.json`  
- **Cell 3**: สร้าง DataFrame และทำความสะอาดข้อมูล (Clean Data)

#### **ขั้นที่ 2️⃣ - สร้าง Dictionary สำหรับจัดกลุ่ม:**
- **Cell 4**: สร้าง Dictionary หลัก (job_categories, benefits, locations, etc.)
- **Cell 5**: สร้าง Dictionary เพิ่มเติมสำหรับการจัดกลุ่ม
- **Cell 6**: สร้าง Dictionary สำหรับ Thai language support

#### **ขั้นที่ 3️⃣ - จัดกลุ่มและวิเคราะห์ข้อมูล:**
- **Cell 7**: ฟังก์ชันสำหรับจัดกลุ่ม + ใช้กับ DataFrame (⚠️ this is where error occurs)
- **Cell 7B**: ฟังก์ชันเพิ่มเติมสำหรับจัดกลุ่ม
- **Cell 7C**: Thai translation functions
- **Cell 7D**: Thai-specific dictionaries

#### **ขั้นที่ 4️⃣ - สะสุมผลและบันทึก:**
- **Cell 8**: แสดงสรุปข้อมูล (Summary)
- **Cell 9**: บันทึกไฟล์ CSV + JSON
- **Cell 10**: แสดงตัวอย่างข้อมูล

### 🚀 **วิธีรัน:**
1. **รัน All Cells:** `Ctrl + Shift + Enter` หรือ `Ctrl + A` → `Shift + Enter`
2. **รัน Cells ทีละตัว:** เลือก Cell แล้ว `Ctrl + Enter`
3. **รัน Cell ถัดไป:** `Alt + Shift + Enter`

In [20]:
output_filename = 'all_files_combined.json'

# 2. ทำการ Save ไฟล์ (เขียนไฟล์ลงเครื่อง)
with open(output_filename, 'w', encoding='utf-8') as f:
    # indent=4 จะช่วยจัดหน้าตา JSON ให้เว้นบรรทัดและอ่านง่ายขึ้น (Pretty Print)
    # ensure_ascii=False ทำให้รองรับภาษาไทย ไม่กลายเป็นตัวอักษรยึกยือ
    json.dump(combined_data, f, ensure_ascii=False, indent=4)

print(f"บันทึกไฟล์ {output_filename} สำเร็จเรียบร้อย!")

บันทึกไฟล์ all_files_combined.json สำเร็จเรียบร้อย!


In [21]:

import pandas as pd
import re
from collections import Counter

# โหลดข้อมูลจากไฟล์ JSON ที่รวมไว้
with open('all_files_combined.json', 'r', encoding='utf-8') as f:
    all_jobs = json.load(f)

# สร้าง DataFrame จากข้อมูล
df = pd.DataFrame(all_jobs)

print("🔍 ตรวจสอบโครงสร้างข้อมูล:")
print(f"   📊 จำนวนแถว: {len(df)}")
print(f"   📋 คอลัมน์ที่รับ: {list(df.columns)}")
print(f"\n   ❓ ตัวอย่างข้อมูลแถวแรก:")
if len(df) > 0:
    print(f"      {df.iloc[0].to_dict()}")

# 🔧 ค้นหาคอลัมน์ที่มีข้อมูลชื่องาน, บริษัท และคำอธิบาย
# การตั้งชื่อคอลัมน์อาจแตกต่างกันในแต่ละไฟล์ JSON
title_cols = [col for col in df.columns if any(name in col.lower() for name in ['title', 'name', 'position', 'job'])]
company_cols = [col for col in df.columns if any(name in col.lower() for name in ['company', 'employer', 'org'])]
desc_cols = [col for col in df.columns if any(name in col.lower() for name in ['description', 'desc', 'requirement', 'detail'])]

print(f"\n   🎯 คอลัมน์ที่อาจเกี่ยวข้อง:")
print(f"      Job Title: {title_cols}")
print(f"      Company: {company_cols}")
print(f"      Description: {desc_cols}")

# เลือกคอลัมน์ที่จะใช้ (ถ้าหลายตัว ให้เลือกตัวแรก)
title_col = title_cols[0] if title_cols else None
company_col = company_cols[0] if company_cols else None
desc_col = desc_cols[0] if desc_cols else None

if not all([title_col, company_col, desc_col]):
    print("\n⚠️  ไม่เจอคอลัมน์ที่จำเป็น!")
    print(f"   Title: {title_col}, Company: {company_col}, Description: {desc_col}")
    import sys
    sys.exit()

print(f"\n   ✅ ใช้คอลัมน์:")
print(f"      Title: {title_col}")
print(f"      Company: {company_col}")
print(f"      Description: {desc_col}")

# 🧹 ทำความสะอาดข้อมูล: ลบแถวที่ไม่สมบูรณ์
required_columns = [title_col, company_col, desc_col]
df = df.dropna(subset=required_columns)

# ลบแถวที่ซ้ำกัน
df = df.drop_duplicates(subset=required_columns)

# 🔧 ฟังก์ชันสำหรับทำความสะอาด Description
def clean_job_description(desc):
    """ลบข้อความขยะและทำความสะอาด"""
    try:
        # แปลงเป็นสตริง
        desc = str(desc).lower()
        # ลบข้อความแปลกประหลาดและอักษรพิเศษ
        desc = desc.replace('\\n', ' ').replace('\\r', ' ').replace('\\t', ' ')
        # ลบ HTML tags ถ้ามี
        desc = re.sub(r'<[^>]+>', '', desc)
        return desc.strip()
    except:
        return ""

# 🎯 ฟังก์ชันเพื่อ Extract Technical Skills (120+ skills)
def extract_skills(description):
    """ดึง Technical Skills จากคำอธิบายงาน"""
    skills_list = [
        'python', 'java', 'c#', 'c++', 'javascript', 'typescript', 'php', 'ruby', 'go', 'rust',
        'react', 'vue', 'angular', 'svelte', 'html', 'css', 'scss', 'less',
        'nodejs', 'express', 'django', 'flask', 'fastapi', 'spring', 'spring boot', 'laravel', 'rails',
        'sql', 'mysql', 'postgresql', 'oracle', 'mongodb', 'redis', 'elasticsearch', 'cassandra',
        'aws', 'azure', 'gcp', 'google cloud', 'heroku', 'firebase',
        'docker', 'kubernetes', 'jenkins', 'gitlab', 'github', 'bitbucket', 'circleci',
        'git', 'svn', 'mercurial', 'graphql', 'rest', 'soap',
        'linux', 'windows', 'macos', 'ubuntu', 'centos',
        'agile', 'scrum', 'kanban', 'microservices', 'api', 'soap',
        'machine learning', 'deep learning', 'tensorflow', 'pytorch', 'scikit-learn', 'keras',
        'spark', 'hadoop', 'hive', 'pig', 'kafka', 'rabbitmq', 'zookeeper',
        'tableau', 'power bi', 'qlik', 'looker', 'metabase',
        'ci/cd', 'devops', 'iac', 'terraform', 'ansible', 'puppet', 'chef',
        'junit', 'pytest', 'mocha', 'jest', 'selenium', 'cypress',
        'webpack', 'gulp', 'gradle', 'maven', 'npm', 'yarn', 'pip', 'composer',
        'jira', 'confluence', 'asana', 'monday', 'trello',
        'figma', 'adobe xd', 'sketch', 'invision',
        'salesforce', 'sap', 'oracle crm', 'dynamics', 'netsuite',
        'ios', 'android', 'flutter', 'react native', 'swift', 'kotlin',
        'solidity', 'web3', 'ethereum', 'blockchain',
    ]
    
    description_lower = str(description).lower()
    found_skills = [skill for skill in skills_list if skill in description_lower]
    return ', '.join(found_skills) if found_skills else 'No specific skills mentioned'

# ✨ ใช้ฟังก์ชัน
df['Full_Description'] = df[desc_col].apply(clean_job_description)
df['Skills'] = df['Full_Description'].apply(extract_skills)

# ✅ ปรับชื่อคอลัมน์ให้สอดคล้อง
df = df.rename(columns={
    title_col: 'Job_Title',
    company_col: 'Company',
    desc_col: 'Job_Description'
})

print(f"\n✅ สร้าง DataFrame เสร็จสิ้น")
print(f"   📊 จำนวนแถว (หลังทำความสะอาด): {len(df):,}")
print(f"   📋 จำนวนคอลัมน์: {len(df.columns)}")
print(f"\n   คอลัมน์ที่มีอยู่:")
for i, col in enumerate(df.columns, 1):
    print(f"      {i:2d}. {col}")



🔍 ตรวจสอบโครงสร้างข้อมูล:
   📊 จำนวนแถว: 3556
   📋 คอลัมน์ที่รับ: ['Keyword', 'Job_Title', 'Company', 'Full_Description', 'Job_Link', 'Source_Page']

   ❓ ตัวอย่างข้อมูลแถวแรก:
      {'Keyword': 'jobs', 'Job_Title': 'Full Stack Developer (C#, .Net Core, React, Vue.js) - Hybrid', 'Company': 'CHOCO CARD ENTERPRISE CO., LTD.', 'Full_Description': "Work location: Near BTS Punnawithi/Bang Chak\n\n\nJob Roles\n\nPlan, design, and develop web applications for marketing purposes, using React Framework for Front-End development and .NET Core for Back-End, working with both dynamic and non-dynamic structures\n\nTroubleshoot and resolve system-related issues\n\nCollaborate with relevant teams to understand project requirements and ensure systems are developed in line with company guidelines and best practices\n\nApply modern software development principles to enhance system performance and efficiency\n\nManage documentation related to the developed systems\n\n\n\n\nRequirement:\n\nBachelor's degr

In [22]:
# ---------------------------------------------------------
# 6. สร้าง Dictionary สำหรับการจัดกลุ่มและประเมินข้อมูล
# ---------------------------------------------------------

# 🏢 Dictionary สำหรับจัดกลุ่มประเภทงาน
job_categories = {
    'Development': ['developer', 'programmer', 'engineer', 'full stack', 'backend', 'frontend', 'นักพัฒนา', 'โปรแกรมเมอร์', 'วิศวกร', 'ผู้พัฒนา', 'ดีเวลอปเปอร์'],
    'Data & Analytics': ['data engineer', 'data scientist', 'analyst', 'bi developer', 'data architect', 'วิศวกรข้อมูล', 'นักวิทยาศาสตร์ข้อมูล', 'นักวิเคราะห์', 'ผู้เชี่ยวชาญข้อมูล'],
    'DevOps & Infrastructure': ['devops', 'sre', 'infrastructure', 'system admin', 'cloud engineer', 'ผู้ดูแล', 'ระบบ', 'เซิร์ฟเวอร์', 'วิศวกรระบบก', 'ผู้ดูแลระบบ'],
    'Quality & Testing': ['qa', 'tester', 'quality assurance', 'automation tester', 'ทดสอบ', 'ควบคุมคุณภาพ', 'ผู้ทดสอบ', 'ผู้ประเมินคุณภาพ'],
    'Database Administration': ['dba', 'database administrator', 'data management', 'ฐานข้อมูล', 'ดีบีเอ', 'ผู้ดูแลฐานข้อมูล'],
    'Business & Management': ['manager', 'lead', 'director', 'head of', 'project manager', 'business', 'ผู้จัดการ', 'ผู้บริหาร', 'หัวหน้า', 'ผู้โครงการ', 'ผู้จัดการโครงการ'],
    'Teaching & Training': ['teacher', 'instructor', 'trainer', 'tutor', 'lecturer', 'อาจารย์', 'ครู', 'เทรนเนอร์', 'ผู้สอน', 'สอนตัวต่อตัว'],
    'Sales & Business Dev': ['sales', 'business development', 'account executive', 'sales manager', 'ขาย', 'พัฒนาธุรกิจ', 'ตัวแทนขาย', 'ผู้มีอำนาจจำหน่าย'],
    'Other': []
}

# 💼 Dictionary สำหรับจัดกลุ่มประเภทสวัสดิการ
benefits_keywords = {
    'Annual Leave': ['annual leave', 'paid vacation', 'vacation days', 'day off', 'วันลาประจำปี', 'ลาพักร้อน', 'วันลา', 'วันพักผ่อน', 'หยุดประจำปี'],
    'Health & Insurance': ['insurance', 'health', 'medical', 'accident insurance', 'healthcare', 'ประกัน', 'สุขภาพ', 'ประกันอุบัติเหตุ', 'ประกันสุขภาพ', 'การรักษา'],
    'Bonus & Incentive': ['bonus', 'performance bonus', 'incentive', 'commission', 'โบนัส', 'เบี้ยประสิทธิภาพ', 'เบี้ยสรุปผล', 'สิ่งจูงใจ'],
    'Flexibility': ['flexible', 'work from home', 'hybrid', 'remote', 'ยืดหยุ่น', 'ทำงานจากบ้าน', 'ระยะไกล', 'ทำงานจากที่บ้าน', 'วฟเอช'],
    'Learning & Development': ['training', 'development', 'workshop', 'seminar', 'e-learning', 'อบรม', 'พัฒนา', 'สัมมนา', 'การศึกษา', 'การเรียนรู้'],
    'Team Activities': ['team building', 'company activities', 'social', 'outing', 'party', 'กิจกรรมทีม', 'ปฐมนิเทศ', 'วิทยาทัศน์', 'ทริปต่างจังหวัร'],
    'Allowance': ['allowance', 'meal', 'transportation', 'mobile', 'housing', 'เบี้ยต่าง', 'ค่าอาหาร', 'ค่าเดินทาง', 'ค่าโทรศัพท์', 'ค่าที่อยู่'],
    'Retirement': ['provident fund', 'pension', '401k', 'กองทุนสัญญาณ', 'บำเหน็จ', 'กองทุนประชาชน', 'สุขสวัสดิการ']
}

# 📍 Dictionary สำหรับจัดกลุ่มสถานที่ทำงาน
location_keywords = {
    'Bangkok': ['bangkok', 'bts', 'mrt', 'rama', 'sukhumvit', 'silom', 'ratchadamri', 'sathorn', 'গรุงเทพ', 'บีทีเอส', 'เอ็มอาร์ที', 'ลาดพร้าว', 'พระโขนง'],
    'Central': ['ayutthaya', 'lopburi', 'nakhon pathom', 'kanchanaburi', 'อยุธยา', 'ลพบุรี', 'นครปฐม', 'กาญจนบุรี'],
    'Eastern': ['chonburi', 'rayong', 'chachoengsao', 'trad', 'pattaya', 'ชลบุรี', 'ระยอง', 'พัทยา', 'ฉะเชิงเทรา', 'ตราด'],
    'Northern': ['chiang mai', 'chiang rai', 'phrae', 'nan', 'phayao', 'เชียงใหม่', 'เชียงราย', 'พะเยา', 'น่าน', 'แพร่'],
    'Northeastern': ['khon kaen', 'udon thani', 'nakhon ratchasima', 'roi et', 'ubon', 'ขอนแก่น', 'อุดรธานี', 'นครราชสีมา', 'ร้อยเอ็ด', 'อุบลราชธานี'],
    'Southern': ['phuket', 'pattani', 'songkhla', 'satun', 'krabi', 'phang nga', 'ภูเก็ต', 'สงขลา', 'กระบี่', 'ตรัง', 'สตูล'],
    'Western': ['phetchaburi', 'prachuap khiri khan', 'ranong', 'chumphon', 'เพชรบูรณ์', 'ประจวบคีรีขันธ์', 'ระนอง', 'ชุมพร'],
    'Hybrid': ['hybrid', 'work from home', 'remote', 'ผสมผสาน', 'ทำงานจากบ้าน', 'ยืดหยุ่น', 'วฟเอช', 'บ้านและสำนักงาน'],
    'On-site': ['on-site', 'office', 'สำนักงาน', 'เข้างาน', 'ที่สำนักงาน', 'ในสถานที่']
}

# 👨‍💼 Dictionary สำหรับจัดกลุ่มสถานะของผู้สมัคร
experience_levels = {
    'Entry Level': ['junior', 'fresh', 'graduate', 'intern', '0-1 years', '0-2 years', 'จบใหม่', 'fang', 'เข้าใหม่', 'จูเนียร์', 'ฝึกงาน'],
    'Mid Level': ['2-5 years', '3-5 years', 'mid-level', 'senior junior', '5+ years', 'ปานกลาง', 'กลางคัน', 'มีประสบการณ์'],
    'Senior Level': ['senior', '10+ years', 'lead', 'principal', 'สูง', 'ชำนาญการ', 'อาวุโส', 'เมืองหลวง'],
    'Management': ['manager', 'director', 'head', 'supervisor', 'team lead', 'ผู้บริหาร', 'ผู้จัดการ', 'ผู้นำชุด', 'เจ้าภาพ']
}

# 💰 Dictionary สำหรับประเมินระดับเงินเดือน (โดยประมาณ)
salary_ranges = {
    'Entry': [0, 30000],
    'Low': [30000, 50000],
    'Mid': [50000, 80000],
    'High': [80000, 150000],
    'Very High': [150000, 250000],
    'Executive': [250000, float('inf')]
}

# 🔧 Dictionary สำหรับจัดกลุ่มธรรมชาติของบริษัท
company_types = {
    'Financial': ['bank', 'finance', 'insurance', 'trading', 'investment', 'securities', 'ธนาคาร', 'เงิน', 'ประกัน', 'การลงทุน', 'หลักทรัพย์'],
    'Technology': ['tech', 'software', 'it', 'digital', 'startup', 'saas', 'เทค', 'ดีจิทัล', 'เทคโนโลยี', 'ซอฟต์แวร์', 'ไอที'],
    'Manufacturing': ['factory', 'manufacturing', 'automotive', 'electronics', 'production', 'โรงงาน', 'การผลิต', 'รถยนต์', 'อิเล็กทรอนิกส์'],
    'Retail & E-commerce': ['retail', 'e-commerce', 'shopping', 'mall', 'trade', 'ค้าปลีก', 'การค้าออนไลน์', 'ร้านค้า', 'ห้างสรรพสินค้า'],
    'Telecom & Media': ['telecom', 'media', 'broadcasting', 'communication', 'โทรคมนาคม', 'สื่อ', 'สิ่งพิมพ์', 'ข่าว'],
    'Energy & Utilities': ['energy', 'power', 'utility', 'oil', 'gas', 'พลังงาน', 'ไฟฟ้า', 'น้ำประปา', 'ก้าซ', 'น้ำมัน'],
    'Real Estate & Construction': ['real estate', 'construction', 'property', 'development', 'อสังหาริมทรัพย์', 'ก่อสร้าง', 'โครงการ', 'สถาปัตยกรรม'],
    'Hospitality & Tourism': ['hotel', 'restaurant', 'tourism', 'aviation', 'hospitality', 'โรงแรม', 'ร้านอาหาร', 'ท่องเที่ยว', 'ตัวแทนเที่ยว'],
    'Healthcare': ['hospital', 'clinic', 'pharma', 'healthcare', 'medical', 'โรงพยาบาล', 'สถานพยาบาล', 'เภสัชกรรม', 'สุขภาพ'],
    'Education': ['school', 'university', 'education', 'institute', 'academy', 'โรงเรียน', 'มหาวิทยาลัย', 'สถาบัน', 'วิทยาลัย']
}

# ===== 🆕 NEW DICTIONARIES WITH THAI =====

# 💻 Dictionary สำหรับจัดกลุ่มประเภทการจ้างงาน
job_type_keywords = {
    'Full-time': ['full-time', 'fulltime', 'permanent', 'permanent position', 'ประจำ', 'ตลอดเวลา', 'เต็มเวลา', 'ประจำตำแหน่ง'],
    'Part-time': ['part-time', 'parttime', 'part time', 'นอกเวลา', 'เศษเวลา'],
    'Contract': ['contract', 'fixed-term', 'fixed term', 'สัญญา', 'ตามสัญญา', 'จ้างตามสัญญา'],
    'Internship': ['intern', 'internship', 'co-op', 'apprenticeship', 'ฝึกงาน', 'เทรนนี่', 'ฝึกอบรม'],
    'Temporary': ['temporary', 'freelance', 'project-based', 'occasional', 'ชั่วคราว', 'อิสระ', 'ตามโครงการ'],
    'Probation': ['probation', 'trial period', 'สอบเลือก', 'ทดลองงาน', 'วิทยาทัศน์']
}

# 🎓 Dictionary สำหรับระดับการศึกษาที่ต้องการ
education_level_keywords = {
    'High School': ['high school', 'matayom', 'ม.ปลาย', 'secondary', 'มัธยม', 'มัธยมศึกษา'],
    'Diploma': ['diploma', 'vocational', 'ปวช', 'ปวส', 'ประกาศนียบัตร', 'อาชีวศึกษา'],
    'Bachelor': ['bachelor', 'bachelor degree', 'degree', 'ปริญญาตรี', 'ป.ตรี', 'ปริญญา'],
    'Master': ['master', 'masters degree', 'm.s', 'ม.บ', 'ปริญญาโท', 'ป.โท'],
    'PhD': ['phd', 'doctoral', 'ph.d', 'doctorate', 'ปริญญาเอก', 'ป.เอก', 'ปริญญาเอกแบบบูรณาการ'],
    'Any': ['no specific', 'any education', 'จบการศึกษา', 'อื่นๆ', 'ไม่จำกัด']
}

# 🌍 Dictionary สำหรับภาษาที่ต้องการ
language_keywords = {
    'English': ['english', 'english proficiency', 'english speaking', 'good english', 'ภาษาอังกฤษ', 'อังกฤษดี', 'พูดอังกฤษ'],
    'Thai': ['thai', 'language thai', 'fluent thai', 'ภาษาไทย', 'ไทย', 'ภาษาแม่'],
    'Mandarin': ['mandarin', 'chinese', 'simplified chinese', 'ภาษาจีน', 'จีน', 'ภาษาจีนกลาง'],
    'Japanese': ['japanese', 'nihongo', 'ญี่ปุ่น', 'ภาษาญี่ปุ่น', 'ญี่ปุ่นดี'],
    'Korean': ['korean', 'เกาหลี', 'ภาษาเกาหลี'],
    'German': ['german', 'deutsch', 'เยอรมัน', 'เยอรมันดี'],
    'French': ['french', 'ฝรั่งเศส', 'ภาษาฝรั่งเศส'],
    'Multilingual': ['multilingual', 'bilingual', 'language', 'fluent in both', 'หลายภาษา', 'ทวิภาษา']
}

# 💪 Dictionary สำหรับ Soft Skills ที่ต้องการ
soft_skills_keywords = {
    'Communication': ['communication', 'communicative', 'presenting', 'presentation', 'verbal skills', 'การสื่อสาร', 'นำเสนอ', 'สื่อสารที่ดี'],
    'Leadership': ['leadership', 'leader', 'leading', 'motivate', 'ภาวะผู้นำ', 'นำทีม', 'ผู้นำ'],
    'Problem-solving': ['problem solving', 'problem-solving', 'analytical', 'critical thinking', 'การแก้ปัญหา', 'วิเคราะห์', 'คิดวิเคราะห์'],
    'Teamwork': ['team', 'teamwork', 'collaboration', 'team player', 'collaborative', 'ทำงานเป็นทีม', 'ร่วมมือ'],
    'Adaptability': ['adaptive', 'adaptable', 'flexible', 'able to adapt', 'quick learner', 'ทำงานได้ยืดหยุ่น', 'เรียนรู้เร็ว', 'ปรับตัวเร็ว'],
    'Time Management': ['time management', 'deadline', 'organized', 'punctual', 'จัดการเวลา', 'ตรงต่อเวลา', 'จัดระเบียบ'],
    'Creativity': ['creative', 'innovation', 'innovative', 'creative thinking', 'สร้างสรรค์', 'นวัตกรรม', 'ความเป็นสร้างสรรค์'],
    'Customer Service': ['customer service', 'customer-oriented', 'client-focused', 'customer care', 'บริการลูกค้า', 'ใจบริการ', 'มีมนุษยสัมพันธ์']
}

# 🛠 Dictionary สำหรับ Technical Stack Groups
tech_stack_groups = {
    'Frontend': ['react', 'vue', 'angular', 'html', 'css', 'javascript', 'typescript', 'flutter', 'swift', 'kotlin'],
    'Backend': ['python', 'java', 'c#', '.net', 'php', 'nodejs', 'golang', 'ruby', 'spring', 'django'],
    'Database': ['sql', 'mysql', 'postgresql', 'oracle', 'mongodb', 'redis', 'nosql', 'cassandra'],
    'Cloud': ['aws', 'azure', 'gcp', 'kubernetes', 'docker', 'heroku', 'firebase'],
    'DevOps': ['docker', 'kubernetes', 'jenkins', 'gitlab', 'github', 'ci/cd', 'terraform', 'ansible'],
    'Data': ['python', 'r', 'spark', 'hadoop', 'kafka', 'tableau', 'power bi', 'machine learning'],
    'Mobile': ['ios', 'android', 'react native', 'flutter', 'swift', 'kotlin']
}

# 📋 Dictionary สำหรับประเภทฝ่ายงาน/ทีม
department_keywords = {
    'Engineering': ['engineering', 'development', 'technical', 'วิศวกรรม', 'พัฒนา', 'วิศวกร', 'ทีมเทคนิค'],
    'Product': ['product', 'product manager', 'pm', 'ผลิตภัณฑ์', 'ผู้จัดการผลิตภัณฑ์'],
    'IT/Infrastructure': ['it', 'infrastructure', 'network', 'system', 'ไอที', 'ระบบ', 'เครือข่าย'],
    'HR': ['hr', 'human resources', 'recruitment', 'talent', 'ทรัพยากรบุคคล', 'ทีมประจำสัญญา', 'บุคลากร'],
    'Sales/Marketing': ['sales', 'marketing', 'business development', 'ขาย', 'การตลาด', 'ผู้ขาย', 'ทีมขาย'],
    'Finance/Accounting': ['finance', 'accounting', 'audit', 'controller', 'การเงิน', 'บัญชี', 'ตรวจสอบ'],
    'Operations': ['operations', 'supply chain', 'logistics', 'ปฏิบัติการ', 'อปก', 'โลจิสติก'],
    'Quality': ['quality', 'qa', 'testing', 'ควบคุมคุณภาพ', 'ทดสอบ', 'ทีมควบคุมคุณภาพ'],
    'Customer Support': ['customer support', 'customer service', 'help desk', 'สนับสนุนลูกค้า', 'บริการ', 'คำอธิบายสอน']
}

# 📝 Dictionary สำหรับประเภทความรับผิดชอบ
responsibility_keywords = {
    'Development': ['develop', 'programming', 'coding', 'implementation', 'build', 'พัฒนา', 'เขียนโปรแกรม', 'สร้าง'],
    'Maintenance': ['maintain', 'maintenance', 'support', 'troubleshoot', 'ดูแลรักษา', 'สนับสนุน', 'ซ่อมแซม'],
    'Planning': ['planning', 'strategic', 'plan', 'roadmap', 'วางแผน', 'กลยุทธ์', 'แผนการ'],
    'Analysis': ['analysis', 'analyze', 'requirement', 'feasibility', 'วิเคราะห์', 'สำรวจ', 'ศึกษา'],
    'Monitoring': ['monitor', 'monitoring', 'tracking', 'observe', 'ตรวจสอบ', 'เฝ้าติดตาม', 'ติดตาม'],
    'Documentation': ['documentation', 'document', 'เอกสาร', 'บันทึก', 'ราคาเอกสาร'],
    'Collaboration': ['collaborate', 'work with', 'coordinate', 'liaise', 'ความร่วมมือ', 'ร่วมมือ'],
    'Training': ['training', 'mentor', 'mentoring', 'guide', 'teach', 'อบรม', 'สอน', 'ให้คำปรึกษา']
}

# 🎯 Dictionary สำหรับการสมัครงาน Application Method
application_method_keywords = {
    'Quick Apply': ['quick apply', 'quickapply', 'สมัครด่วน', 'สมัครทันที'],
    'Email': ['email', 'send resume', 'submit email', 'อีเมล', 'ส่งเมล', 'ส่งจดหมาย'],
    'Direct Application': ['apply online', 'apply now', 'application form', 'สมัครออนไลน์', 'แบบฟอร์ม'],
    'Phone/Contact': ['contact', 'phone', 'call', 'reach out', 'ติดต่อ', 'โทร', 'ติดต่อโดยตรง'],
    'Upload Required': ['upload', 'attach', 'submit documents', 'อัปโหลด', 'ส่งไฟล์', 'แนบเอกสาร']
}

# 📊 Dictionary สำหรับประเมินค่าของตำแหน่ง
position_seniority = {
    'Entry Level': ['junior', 'graduate', 'trainee', 'assistant', 'executive - trainee', 'จูเนียร์', 'ผู้ช่วย', 'เข้าใหม่'],
    'Mid-Career': ['senior', 'officer', 'specialist', '3-5 years', 'ชำนาญการ', 'เจ้าหน้าที่', 'ผู้เชี่ยวชาญ'],
    'Advanced': ['lead', 'principal', 'staff', '7-10 years', 'หัวหน้า', 'นำทีม', 'อาวุโส'],
    'Management': ['manager', 'senior manager', 'director', 'head of', 'ผู้จัดการ', 'ผู้บริหาร', 'ผู้บริหารระดับสูง'],
    'Executive': ['ceo', 'cto', 'cfo', 'vp', 'vice president', 'executive', 'ผู้บริหารสูง', 'ผู้บริหารสูงสุด']
}

# ⏰ Dictionary สำหรับรูปแบบการทำงาน
work_arrangement_keywords = {
    'Office': ['office', 'on-site', 'on site', 'สำนักงาน', 'เข้างาน', 'ที่สำนักงาน', 'ที่ปฏิบัติงาน'],
    'Remote': ['remote', 'work from home', 'wfh', 'home-based', 'ทำงานจากบ้าน', 'ระยะไกล', 'วฟเอช'],
    'Hybrid': ['hybrid', 'ผสมผสาน', 'ผสม', 'บ้านและสำนักงาน', 'ยืดหยุ่น'],
    'Flexible Hours': ['flexible hours', 'flexible time', 'flexible', 'ยืดหยุ่น', 'เวลาอิสระ', 'เวลางานยืดหยุ่น'],
    'Regular Hours': ['regular hours', 'standard hours', '9-5', '9 am to 5 pm', 'เวลาประจำ', 'เวลาราชการ'],
    'Shift Work': ['shift', 'rotating', '24/7', 'weekend', 'เสมา', 'อาทิตย์', 'กะเวรรค์', 'กะการทำงาน']
}

# 🏆 Dictionary สำหรับหน้าที่หลัก
primary_function_keywords = {
    'Coding/Development': ['code', 'develop', 'programming', 'developer', 'เขียนโปรแกรม', 'นักพัฒนา', 'พัฒนาระบบ'],
    'Management': ['manage', 'management', 'lead', 'oversee', 'บริหาร', 'จัดการ', 'ดูแล'],
    'Design': ['design', 'designer', 'ui', 'ux', 'ออกแบบ', 'ออกแบบกราฟิก', 'ผู้ออกแบบ'],
    'Testing': ['test', 'qa', 'quality', 'ทดสอบ', 'ควบคุมคุณภาพ', 'ผู้ทดสอบ'],
    'Sales': ['sales', 'selling', 'business development', 'ขาย', 'พัฒนาธุรกิจ', 'ตัวแทนขาย'],
    'Support': ['support', 'customer service', 'help desk', 'สนับสนุน', 'บริการ', 'ช่วยเหลือ'],
    'Analysis': ['analysis', 'analytical', 'data', 'วิเคราะห์', 'ข้อมูล', 'วิเคราะห์ข้อมูล'],
    'Operations': ['operations', 'operational', 'ปฏิบัติการ', 'อปก', 'ดำเนินการ']
}

print("✅ สร้าง Dictionary สำหรับการจัดกลุ่มข้อมูลเสร็จสิ้น")
print(f"   - Job Categories: {len(job_categories)} ประเภท")
print(f"   - Benefits Keywords: {len(benefits_keywords)} ประเภท")
print(f"   - Location Keywords: {len(location_keywords)} ประเภท")
print(f"   - Experience Levels: {len(experience_levels)} สถานะ")
print(f"   - Salary Ranges: {len(salary_ranges)} ระดับ")
print(f"   - Company Types: {len(company_types)} ประเภท")
print(f"\n   🆕 NEW DICTIONARIES ADDED (WITH THAI LANGUAGE SUPPORT):")
print(f"   - Job Type: {len(job_type_keywords)} ประเภท")
print(f"   - Education Level: {len(education_level_keywords)} ระดับ")
print(f"   - Language: {len(language_keywords)} ภาษา")
print(f"   - Soft Skills: {len(soft_skills_keywords)} ทักษะ")
print(f"   - Tech Stack Groups: {len(tech_stack_groups)} กลุ่ม")
print(f"   - Department: {len(department_keywords)} ฝ่าย")
print(f"   - Responsibility Keywords: {len(responsibility_keywords)} ประเภท")
print(f"   - Application Method: {len(application_method_keywords)} วิธี")
print(f"   - Position Seniority: {len(position_seniority)} ระดับ")
print(f"   - Work Arrangement: {len(work_arrangement_keywords)} รูปแบบ")
print(f"   - Primary Function: {len(primary_function_keywords)} หน้าที่")

✅ สร้าง Dictionary สำหรับการจัดกลุ่มข้อมูลเสร็จสิ้น
   - Job Categories: 9 ประเภท
   - Benefits Keywords: 8 ประเภท
   - Location Keywords: 9 ประเภท
   - Experience Levels: 4 สถานะ
   - Salary Ranges: 6 ระดับ
   - Company Types: 10 ประเภท

   🆕 NEW DICTIONARIES ADDED (WITH THAI LANGUAGE SUPPORT):
   - Job Type: 6 ประเภท
   - Education Level: 6 ระดับ
   - Language: 8 ภาษา
   - Soft Skills: 8 ทักษะ
   - Tech Stack Groups: 7 กลุ่ม
   - Department: 9 ฝ่าย
   - Responsibility Keywords: 8 ประเภท
   - Application Method: 5 วิธี
   - Position Seniority: 5 ระดับ
   - Work Arrangement: 6 รูปแบบ
   - Primary Function: 8 หน้าที่


In [23]:
# ===== 🌐 Thai-English Benefits Mapping =====
benefits_mapping = {
    'Annual Leave': {
        'thai': 'วันลาประจำปี',
        'keywords_en': ['annual leave', 'paid vacation', 'vacation days', 'day off', 'holidays'],
        'keywords_th': ['วันลาประจำปี', 'ลาพักร้อน', 'วันลา', 'วันพักผ่อน', 'หยุดประจำปี', 'ลาพัก']
    },
    'Health & Insurance': {
        'thai': 'ประกัน & สุขภาพ',
        'keywords_en': ['insurance', 'health', 'medical', 'accident insurance', 'healthcare', 'health insurance', 'opd'],
        'keywords_th': ['ประกัน', 'สุขภาพ', 'ประกันอุบัติเหตุ', 'ประกันสุขภาพ', 'การรักษา', 'ประกันกลุ่ม']
    },
    'Bonus & Incentive': {
        'thai': 'โบนัส & สิ่งจูงใจ',
        'keywords_en': ['bonus', 'performance bonus', 'incentive', 'commission', '13th month', 'results-based bonus'],
        'keywords_th': ['โบนัส', 'เบี้ยประสิทธิภาพ', 'เบี้ยสรุปผล', 'สิ่งจูงใจ', 'เงินประกอบการ']
    },
    'Flexibility': {
        'thai': 'ความยืดหยุ่น',
        'keywords_en': ['flexible', 'work from home', 'hybrid', 'remote', 'flexible hours', 'wfh'],
        'keywords_th': ['ยืดหยุ่น', 'ทำงานจากบ้าน', 'ระยะไกล', 'ทำงานจากที่บ้าน', 'วฟเอช', 'เวลาอิสระ']
    },
    'Learning & Development': {
        'thai': 'การเรียนรู้ & พัฒนา',
        'keywords_en': ['training', 'development', 'workshop', 'seminar', 'e-learning', 'learning', 'educational'],
        'keywords_th': ['อบรม', 'พัฒนา', 'สัมมนา', 'การศึกษา', 'การเรียนรู้', 'การพัฒนาทักษะ', 'ท่องเที่ยว']
    },
    'Team Activities': {
        'thai': 'กิจกรรมทีม',
        'keywords_en': ['team building', 'company activities', 'social', 'outing', 'party', 'activities'],
        'keywords_th': ['กิจกรรมทีม', 'ปฐมนิเทศ', 'วิทยาทัศน์', 'ทริปต่างจังหวัร', 'ร้อน', 'วสท']
    },
    'Allowance': {
        'thai': 'เบี้ย & ค่าใช้จ่าย',
        'keywords_en': ['allowance', 'meal', 'transportation', 'mobile', 'housing', 'bonus', 'subsidy'],
        'keywords_th': ['เบี้ยต่าง', 'ค่าอาหาร', 'ค่าเดินทาง', 'ค่าโทรศัพท์', 'ค่าที่อยู่', 'เบี้ยพิเศษ', 'ค่าน้ำมัน']
    },
    'Retirement': {
        'thai': 'กองทุนเกษียณ',
        'keywords_en': ['provident fund', 'pension', '401k', 'retirement', 'fund'],
        'keywords_th': ['กองทุนสัญญาณ', 'บำเหน็จ', 'กองทุนประชาชน', 'สุขสวัสดิการ', 'ปกษ']
    }
}

def extract_benefits_bilingual(description):
    """
    ดึงประเภทสวัสดิการจากคำอธิบายงาน (ภาษาไทย+อังกฤษ)
    Returns: comma-separated string of benefits or 'Not specified'
    """
    description_lower = str(description).lower()
    found_benefits = set()
    
    # Search for each benefit type
    for benefit_name, benefit_info in benefits_mapping.items():
        # Check English keywords
        for keyword in benefit_info['keywords_en']:
            if keyword in description_lower:
                found_benefits.add(benefit_name)
                break
        
        # Check Thai keywords (if not already found)
        if benefit_name not in found_benefits:
            for keyword in benefit_info['keywords_th']:
                if keyword in description_lower:
                    found_benefits.add(benefit_name)
                    break
    
    return ', '.join(sorted(found_benefits)) if found_benefits else 'Not specified'

def extract_benefits_with_thai(description):
    """
    ดึงสวัสดิการพร้อมแสดงทั้งภาษาไทยและอังกฤษ
    Returns: dict with 'english' and 'thai' keys
    """
    description_lower = str(description).lower()
    found_benefits_en = []
    found_benefits_th = []
    
    # Search for each benefit type
    for benefit_name, benefit_info in benefits_mapping.items():
        found = False
        
        # Check English keywords
        for keyword in benefit_info['keywords_en']:
            if keyword in description_lower:
                found_benefits_en.append(benefit_name)
                found_benefits_th.append(benefit_info['thai'])
                found = True
                break
        
        # Check Thai keywords (if not already found)
        if not found:
            for keyword in benefit_info['keywords_th']:
                if keyword in description_lower:
                    found_benefits_en.append(benefit_name)
                    found_benefits_th.append(benefit_info['thai'])
                    break
    
    return {
        'english': ', '.join(found_benefits_en) if found_benefits_en else 'Not specified',
        'thai': ', '.join(found_benefits_th) if found_benefits_th else 'ไม่ระบุ'
    }

print("✅ สร้างฟังก์ชันสกัด Benefits (ภาษาไทย+อังกฤษ) เสร็จสิ้น")
print("   📌 ฟังก์ชัน: extract_benefits_bilingual() - แสดงประเภทสวัสดิการ")
print("   📌 ฟังก์ชัน: extract_benefits_with_thai() - แสดง ไทย+อังกฤษ")
print(f"   📊 รวมประเภทสวัสดิการ: {len(benefits_mapping)} ประเภท")
print("\n   ตัวอย่างสวัสดิการที่สกัดได้:")
for i, (benefit, info) in enumerate(list(benefits_mapping.items())[:4], 1):
    print(f"      {i}. {benefit} ({info['thai']})")
    print(f"         EN: {', '.join(info['keywords_en'][:2])}")
    print(f"         TH: {', '.join(info['keywords_th'][:2])}")


✅ สร้างฟังก์ชันสกัด Benefits (ภาษาไทย+อังกฤษ) เสร็จสิ้น
   📌 ฟังก์ชัน: extract_benefits_bilingual() - แสดงประเภทสวัสดิการ
   📌 ฟังก์ชัน: extract_benefits_with_thai() - แสดง ไทย+อังกฤษ
   📊 รวมประเภทสวัสดิการ: 8 ประเภท

   ตัวอย่างสวัสดิการที่สกัดได้:
      1. Annual Leave (วันลาประจำปี)
         EN: annual leave, paid vacation
         TH: วันลาประจำปี, ลาพักร้อน
      2. Health & Insurance (ประกัน & สุขภาพ)
         EN: insurance, health
         TH: ประกัน, สุขภาพ
      3. Bonus & Incentive (โบนัส & สิ่งจูงใจ)
         EN: bonus, performance bonus
         TH: โบนัส, เบี้ยประสิทธิภาพ
      4. Flexibility (ความยืดหยุ่น)
         EN: flexible, work from home
         TH: ยืดหยุ่น, ทำงานจากบ้าน


In [24]:
# ✅ ใช้ฟังก์ชันเหล่านี้กับ DataFrame เพื่อสร้างคอลัมน์ใหม่
print("\n⏳ กำลังจัดกลุ่มข้อมูล...")

# 🔴 ตรวจสอบว่า Full_Description มีหรือไม่
if 'Full_Description' not in df.columns:
    print("⚠️  Full_Description ยังไม่มี - กำลังสร้าง...")
    if 'Job_Description' in df.columns:
        df['Full_Description'] = df['Job_Description'].apply(lambda x: str(x).lower() if pd.notna(x) else "")
    else:
        print("⚠️  ERROR: ไม่เจอ Job_Description column!")
        print(f"   คอลัมน์ที่มี: {list(df.columns)}")
        import sys
        sys.exit()

df['Job_Category'] = df['Job_Title'].apply(categorize_job_title)
df['Benefits'] = df['Full_Description'].apply(extract_benefits_bilingual)
df['Job_Location'] = df['Full_Description'].apply(identify_location)
df['Experience_Level'] = df['Full_Description'].apply(estimate_experience_level)
df['Company_Type'] = df['Company'].apply(identify_company_type)

print("✅ จัดกลุ่มข้อมูลเสร็จสิ้น!")
print(f"   ✨ เพิ่มคอลัมน์ 5 คอลัมน์:")
print(f"      • Job_Category")
print(f"      • Benefits (สกัดจากข้อความจริง ภาษาไทย+อังกฤษ)")
print(f"      • Job_Location")
print(f"      • Experience_Level")
print(f"      • Company_Type")
print(f"\n   📝 ตัวอย่างสวัสดิการจากข้อมูลจริง:")
for i, (benefits, title) in enumerate(zip(df['Benefits'].head(3), df['Job_Title'].head(3)), 1):
    print(f"      {i}. {title[:50]}...")
    print(f"         ✅ Benefits: {benefits}")



⏳ กำลังจัดกลุ่มข้อมูล...
⚠️  Full_Description ยังไม่มี - กำลังสร้าง...
✅ จัดกลุ่มข้อมูลเสร็จสิ้น!
   ✨ เพิ่มคอลัมน์ 5 คอลัมน์:
      • Job_Category
      • Benefits (สกัดจากข้อความจริง ภาษาไทย+อังกฤษ)
      • Job_Location
      • Experience_Level
      • Company_Type

   📝 ตัวอย่างสวัสดิการจากข้อมูลจริง:
      1. Full Stack Developer (C#, .Net Core, React, Vue.js...
         ✅ Benefits: Allowance, Annual Leave, Bonus & Incentive, Flexibility, Health & Insurance, Learning & Development, Team Activities
      2. (Junior Level) Backend Developer...
         ✅ Benefits: Learning & Development
      3. Site Reliability Engineer (SRE) – Cloud & DevOps...
         ✅ Benefits: Health & Insurance


In [25]:
# ---------------------------------------------------------
# 7-Extended. ฟังก์ชันเพิ่มเติมสำหรับจัดกลุ่มข้อมูลใหม่ ✨
# ---------------------------------------------------------

def categorize_job_type(description):
    """แยกประเภทการจ้างงาน (Full-time, Part-time ฯลฯ)"""
    description_lower = str(description).lower()
    for job_type, keywords in job_type_keywords.items():
        if any(keyword in description_lower for keyword in keywords):
            return job_type
    return 'Full-time'  # ค่าเริ่มต้น

def extract_education_level(description):
    """ดึงระดับการศึกษาที่ต้องการ"""
    description_lower = str(description).lower()
    found_levels = []
    for level, keywords in education_level_keywords.items():
        if any(keyword in description_lower for keyword in keywords):
            found_levels.append(level)
    return found_levels[0] if found_levels else 'Not specified'

def extract_languages(description):
    """ดึงภาษาที่ต้องการ"""
    description_lower = str(description).lower()
    found_languages = []
    for language, keywords in language_keywords.items():
        if any(keyword in description_lower for keyword in keywords):
            found_languages.append(language)
    return ', '.join(found_languages) if found_languages else 'Thai only'

def extract_soft_skills(description):
    """ดึง Soft Skills ที่ต้องการ"""
    description_lower = str(description).lower()
    found_skills = []
    for skill, keywords in soft_skills_keywords.items():
        if any(keyword in description_lower for keyword in keywords):
            found_skills.append(skill)
    return ', '.join(found_skills) if found_skills else 'Not specified'

def identify_department(description):
    """ระบุฝ่ายงาน"""
    description_lower = str(description).lower()
    for dept, keywords in department_keywords.items():
        if any(keyword in description_lower for keyword in keywords):
            return dept
    return 'Other'

def extract_work_arrangement(description):
    """ดึงรูปแบบการทำงาน"""
    description_lower = str(description).lower()
    found_arrangements = []
    for arrangement, keywords in work_arrangement_keywords.items():
        if any(keyword in description_lower for keyword in keywords):
            found_arrangements.append(arrangement)
    return ', '.join(found_arrangements) if found_arrangements else 'On-site'

def identify_primary_function(description):
    """ระบุหน้าที่หลัก"""
    description_lower = str(description).lower()
    for function, keywords in primary_function_keywords.items():
        if any(keyword in description_lower for keyword in keywords):
            return function
    return 'Other'

def extract_tech_stack(description):
    """ดึง Technology Stack ที่ใช้"""
    description_lower = str(description).lower()
    tech_found = set()
    for tech_group, technologies in tech_stack_groups.items():
        for tech in technologies:
            if tech in description_lower:
                tech_found.add(tech_group)
    return ', '.join(tech_found) if tech_found else 'Not specified'

print("✅ สร้างฟังก์ชันเพิ่มเติมเสร็จสิ้น")
print("   ฟังก์ชันที่สามารถใช้ได้:")
print("   - categorize_job_type()")
print("   - extract_education_level()")
print("   - extract_languages()")
print("   - extract_soft_skills()")
print("   - identify_department()")
print("   - extract_work_arrangement()")
print("   - identify_primary_function()")
print("   - extract_tech_stack()")

✅ สร้างฟังก์ชันเพิ่มเติมเสร็จสิ้น
   ฟังก์ชันที่สามารถใช้ได้:
   - categorize_job_type()
   - extract_education_level()
   - extract_languages()
   - extract_soft_skills()
   - identify_department()
   - extract_work_arrangement()
   - identify_primary_function()
   - extract_tech_stack()


In [26]:
# ===== 🌐 ตัวอย่างสกัด Benefits แบบทวิภาษา =====
print("\n" + "="*70)
print("🌐 ตัวอย่างการสกัด Benefits แบบทวิภาษา (ไทย+อังกฤษ)")
print("="*70)

# ทดสอบกับงาน 5 อันแรก
sample_benefits = []
for idx, row in df.head(5).iterrows():
    benefits_result = extract_benefits_with_thai(row['Full_Description'])
    sample_benefits.append({
        'title': row['Job_Title'][:40],
        'en': benefits_result['english'],
        'th': benefits_result['thai']
    })

for i, benefit_info in enumerate(sample_benefits, 1):
    print(f"\n📌 ตำแหน่งที่ {i}: {benefit_info['title']}...")
    print(f"   🇺🇸 English: {benefit_info['en']}")
    print(f"   🇹🇭 ไทย:    {benefit_info['th']}")

print("\n✅ Benefits สกัดจากคำอธิบายงานจริง (ทั้งภาษาไทยและอังกฤษ)")



🌐 ตัวอย่างการสกัด Benefits แบบทวิภาษา (ไทย+อังกฤษ)

📌 ตำแหน่งที่ 1: Full Stack Developer (C#, .Net Core, Rea...
   🇺🇸 English: Annual Leave, Health & Insurance, Bonus & Incentive, Flexibility, Learning & Development, Team Activities, Allowance
   🇹🇭 ไทย:    วันลาประจำปี, ประกัน & สุขภาพ, โบนัส & สิ่งจูงใจ, ความยืดหยุ่น, การเรียนรู้ & พัฒนา, กิจกรรมทีม, เบี้ย & ค่าใช้จ่าย

📌 ตำแหน่งที่ 2: (Junior Level) Backend Developer...
   🇺🇸 English: Learning & Development
   🇹🇭 ไทย:    การเรียนรู้ & พัฒนา

📌 ตำแหน่งที่ 3: Site Reliability Engineer (SRE) – Cloud ...
   🇺🇸 English: Health & Insurance
   🇹🇭 ไทย:    ประกัน & สุขภาพ

📌 ตำแหน่งที่ 4: Master Data Management Officer...
   🇺🇸 English: Learning & Development
   🇹🇭 ไทย:    การเรียนรู้ & พัฒนา

📌 ตำแหน่งที่ 5: Data Engineer...
   🇺🇸 English: Learning & Development, Allowance
   🇹🇭 ไทย:    การเรียนรู้ & พัฒนา, เบี้ย & ค่าใช้จ่าย

✅ Benefits สกัดจากคำอธิบายงานจริง (ทั้งภาษาไทยและอังกฤษ)


In [27]:

# -------------------------------------------------
# ตัวอย่างผลลัพธ์ที่คาดหวัง
# -------------------------------------------------
print("\n" + "="*80)
print("📊 ตัวอย่างสวัสดิการที่สกัดจากข้อมูลจริง:")
print("="*80)

# ดูตัวอย่าง
sample_idx = 0
if len(df) > sample_idx:
    print(f"\n🔹 ตำแหน่งที่ 1:")
    print(f"   Job Title: {df.iloc[sample_idx]['Job_Title']}")
    print(f"   Company: {df.iloc[sample_idx]['Company']}")
    print(f"   Benefits found: {df.iloc[sample_idx]['Benefits']}")
    print(f"   Description (first 200 chars):")
    print(f"   {df.iloc[sample_idx]['Full_Description'][:200]}...")

# ดูสรุป
print(f"\n📈 สรุปสวัสดิการ:")
print(f"   - จำนวนงาน: {len(df)}")
print(f"   - งานที่มีสวัสดิการที่ระบุ: {(df['Benefits'] != 'Not specified').sum()}")
print(f"   - ร้อยละ: {((df['Benefits'] != 'Not specified').sum() / len(df) * 100):.1f}%")




📊 ตัวอย่างสวัสดิการที่สกัดจากข้อมูลจริง:

🔹 ตำแหน่งที่ 1:
   Job Title: Full Stack Developer (C#, .Net Core, React, Vue.js) - Hybrid
   Company: CHOCO CARD ENTERPRISE CO., LTD.
   Benefits found: Allowance, Annual Leave, Bonus & Incentive, Flexibility, Health & Insurance, Learning & Development, Team Activities
   Description (first 200 chars):
   work location: near bts punnawithi/bang chak


job roles

plan, design, and develop web applications for marketing purposes, using react framework for front-end development and .net core for back-end,...

📈 สรุปสวัสดิการ:
   - จำนวนงาน: 2499
   - งานที่มีสวัสดิการที่ระบุ: 2182
   - ร้อยละ: 87.3%


In [28]:

# ✅ ใช้ฟังก์ชันเพิ่มเติมกับ DataFrame
print("\n⏳ กำลังเพิ่มคอลัมน์ข้อมูลเพิ่มเติม...")

# 🔴 ตรวจสอบว่า Full_Description มีหรือไม่
if 'Full_Description' not in df.columns:
    print("⚠️  Full_Description ยังไม่มี - ใช้ Job_Description แทน...")
    if 'Job_Description' in df.columns:
        df['Full_Description'] = df['Job_Description'].apply(lambda x: str(x).lower() if pd.notna(x) else "")
    else:
        print("⚠️  ERROR: ไม่เจอ Job_Description column!")
        import sys
        sys.exit()

df['Job_Type'] = df['Full_Description'].apply(categorize_job_type)
df['Education_Required'] = df['Full_Description'].apply(extract_education_level)
df['Languages'] = df['Full_Description'].apply(extract_languages)
df['Soft_Skills'] = df['Full_Description'].apply(extract_soft_skills)
df['Department'] = df['Full_Description'].apply(identify_department)
df['Work_Arrangement'] = df['Full_Description'].apply(extract_work_arrangement)
df['Primary_Function'] = df['Full_Description'].apply(identify_primary_function)
df['Tech_Stack'] = df['Full_Description'].apply(extract_tech_stack)

print("✅ เพิ่มคอลัมน์เสร็จสิ้น!")
print(f"   ✨ เพิ่มอีก 8 คอลัมน์:")
print(f"      • Job_Type")
print(f"      • Education_Required")
print(f"      • Languages")
print(f"      • Soft_Skills")
print(f"      • Department")
print(f"      • Work_Arrangement")
print(f"      • Primary_Function")
print(f"      • Tech_Stack")
print(f"\n   📊 DataFrame ตอนนี้มี {len(df.columns)} คอลัมน์เรียบร้อย!")




⏳ กำลังเพิ่มคอลัมน์ข้อมูลเพิ่มเติม...
✅ เพิ่มคอลัมน์เสร็จสิ้น!
   ✨ เพิ่มอีก 8 คอลัมน์:
      • Job_Type
      • Education_Required
      • Languages
      • Soft_Skills
      • Department
      • Work_Arrangement
      • Primary_Function
      • Tech_Stack

   📊 DataFrame ตอนนี้มี 21 คอลัมน์เรียบร้อย!


In [29]:
# ---------------------------------------------------------
# 7-Extra. ฟังก์ชันการแนวแปลงไทย-อังกฤษและการวิเคราะห์ข้อมูล
# ---------------------------------------------------------

# 📊 Dictionary แปลงระหว่างอังกฤษ-ไทย
thai_english_mapping = {
    'Job_Category': 'หมวดหมู่งาน',
    'Experience_Level': 'ระดับประสบการณ์',
    'Company_Type': 'ประเภทบริษัท',
    'Job_Location': 'สถานที่ทำงาน',
    'Benefits': 'สวัสดิการ',
    'Skills_String': 'ทักษะที่ต้องการ',
    'Job_Type': 'ประเภทการจ้างงาน',
    'Education_Level': 'ระดับการศึกษา',
    'Languages': 'ภาษา',
    'Soft_Skills': 'ทักษะชีวิต',
    'Department': 'ฝ่ายงาน',
    'Work_Arrangement': 'รูปแบบการทำงาน',
    'Primary_Function': 'หน้าที่หลัก',
    'Tech_Stack': 'เทคโนโลยีที่ใช้'
}

# 📖 Dictionary แปลงสถานะ-ระดับต่างๆ
thai_status_mapping = {
    # Job Categories
    'Development': 'พัฒนาซอฟต์แวร์',
    'Data & Analytics': 'ข้อมูลและการวิเคราะห์',
    'DevOps & Infrastructure': 'ดีวอปส์และระบบโครงสร้าง',
    'Quality & Testing': 'การควบคุมคุณภาพและทดสอบ',
    'Database Administration': 'บริหารฐานข้อมูล',
    'Business & Management': 'ธุรกิจและบริหารจัดการ',
    'Teaching & Training': 'การสอนและการฝึกอบรม',
    'Sales & Business Dev': 'ขายและพัฒนาธุรกิจ',
    
    # Experience Levels
    'Entry Level': 'ผู้เริ่มต้น',
    'Mid Level': 'ระดับกลาง',
    'Senior Level': 'ระดับสูง',
    'Management': 'บริหารจัดการ',
    
    # Job Types
    'Full-time': 'ประจำเต็มเวลา',
    'Part-time': 'ประจำนอกเวลา',
    'Contract': 'ตามสัญญา',
    'Internship': 'ฝึกงาน',
    'Temporary': 'ชั่วคราว',
    'Probation': 'ทดลองงาน',
    
    # Work Arrangements
    'Office': 'ที่สำนักงาน',
    'Remote': 'ระยะไกล',
    'Hybrid': 'ผสมผสาน',
    'Flexible Hours': 'เวลาอิสระ',
    'Regular Hours': 'เวลาประจำ',
    
    # Education Levels
    'Bachelor': 'ปริญญาตรี',
    'Master': 'ปริญญาโท',
    'PhD': 'ปริญญาเอก',
    'Diploma': 'ประกาศนียบัตร',
    'High School': 'มัธยมศึกษา',
}

def translate_to_thai(text):
    """แปลงข้อความจากอังกฤษเป็นไทย (ถ้ามีในแม่พิมพ์)"""
    return thai_status_mapping.get(text, text)

def get_thai_label(english_label):
    """ได้รับป้ายกำกับภาษาไทยสำหรับชื่อคอลัมน์"""
    return thai_english_mapping.get(english_label, english_label)

# ฟังก์ชันสำหรับการสำรวจและสรุปข้อมูลแบบสองภาษา
def print_distribution_bilingual(df, column, title_en, title_th, top_n=5):
    """แสดงการกระจายตัวของข้อมูลแบบสองภาษา"""
    print(f"\n{title_en} / {title_th}")
    print("=" * 60)
    distribution = df[column].value_counts().head(top_n)
    for i, (value, count) in enumerate(distribution.items(), 1):
        percentage = (count / len(df)) * 100
        thai_value = translate_to_thai(value)
        display_value = f"{value}" if value == thai_value else f"{value} ({thai_value})"
        print(f"   {i}. {display_value}: {count:,} ({percentage:.1f}%)")

print("✅ สร้างฟังก์ชันแปลงภาษาและการแสดงผลแบบสองภาษาเสร็จสิ้น")
print("   ฟังก์ชันสนับสนุนภาษาไทย:")
print("   - translate_to_thai()")
print("   - get_thai_label()")
print("   - print_distribution_bilingual()")
print("\n📖 แม่พิมพ์แปลภาษา:")
print(f"   - thai_english_mapping: {len(thai_english_mapping)} คู่")
print(f"   - thai_status_mapping: {len(thai_status_mapping)} การแปล")

✅ สร้างฟังก์ชันแปลงภาษาและการแสดงผลแบบสองภาษาเสร็จสิ้น
   ฟังก์ชันสนับสนุนภาษาไทย:
   - translate_to_thai()
   - get_thai_label()
   - print_distribution_bilingual()

📖 แม่พิมพ์แปลภาษา:
   - thai_english_mapping: 14 คู่
   - thai_status_mapping: 28 การแปล


In [30]:
# ---------------------------------------------------------
# 7-Thai-Specific. Dictionary เฉพาะสำหรับตลาดงานไทย 🇹🇭
# ---------------------------------------------------------

# ✨ Dictionary สำหรับประเภทอุตสาหกรรมในไทย (ทั้งภาษาไทย/อังกฤษ)
thai_industries = {
    'Financial & Banking': ['ธนาคาร', 'การเงิน', 'ประกัน', 'หลักทรัพย์', 'bank', 'finance', 'insurance'],
    'Automotive': ['อุตสาหกรรมยานยนต์', 'รถยนต์', 'มอเตอร์ไซค์', 'อากาศยาน', 'automotive'],
    'Petrochemical & Energy': ['ปโตรเคมี', 'พลังงาน', 'ปิโตรเลียม', 'นำเข้า', 'petrochemical'],
    'Electronics & IT': ['อิเล็กทรอนิกส์', 'ไอที', 'อุตสาหกรรมท้องถิ่น', 'electronics'],
    'Textile & Fashion': ['สิ่งทอ', 'เครื่องแต่งกาย', 'ผ้า', 'แฟชั่น', 'textile'],
    'Food & Beverage': ['อาหาร', 'เครื่องดื่ม', 'การผลิตอาหาร', 'food', 'beverage'],
    'Tourism & Hospitality': ['ท่องเที่ยว', 'โรงแรม', 'รีสอร์ท', 'ร้านอาหาร', 'tourism', 'hospitality'],
    'Real Estate & Property': ['อสังหาริมทรัพย์', 'พัฒนาอสังหาริมทรัพย์', 'ก่อสร้าง', 'real estate'],
    'Retail & Distribution': ['ค้าปลีก', 'ห้างสรรพสินค้า', 'ศูนย์การค้า', 'retail'],
    'Healthcare': ['โรงพยาบาล', 'คลินิก', 'ยา', 'สุขภาพ', 'healthcare'],
    'Telecommunications': ['โทรศัพท์', 'อินเทอร์เน็ต', 'ระบบสื่อสาร', 'telecom'],
    'Government & NGO': ['ราชการ', 'องค์กรปกครอง', 'องค์กรสัตวบัญชา', 'องค์กรไม่แสวงหาผลกำไร', 'government'],
}

# 🏙️ Dictionary สำหรับเมืองใหญ่ของไทย (ตลาดงาน)
thai_major_cities = {
    'Bangkok': {
        'Thai': 'กรุงเทพมหานคร',
        'Districts': ['สาทร', 'ราชดำเนิน', 'ธนบุรี', 'ปทุมวัน', 'ห้วยขวาง', 'ลาดพร้าว', 'บาง', 'ดุสิต'],
        'BTS_lines': ['สยาม', 'ชิดลม', 'สลัต', 'ปุณณวิถี'],
        'Job_density': 'High',
    },
    'Chiang Mai': {
        'Thai': 'เชียงใหม่',
        'Industries': ['ท่องเที่ยว', 'เกษตรกรรม', 'หัตถศิลป์', 'การเก่าของเก่า'],
        'Job_density': 'Medium',
    },
    'Pattaya': {
        'Thai': 'พัทยา',
        'Industries': ['ท่องเที่ยว', 'บริการ', 'โรงแรม'],
        'Location': 'ชลบุรี',
        'Job_density': 'Medium',
    },
    'Rayong': {
        'Thai': 'ระยอง',
        'Industries': ['ปโตรเคมี', 'อิเล็กทรอนิกส์', 'อุตสาหกรรมเหล็ก'],
        'Zone': 'Eastern Seaboard',
        'Job_density': 'High',
    },
}

# 💰 Dictionary สำหรับเงินเดือนโดยการจ้างงาน (คร่าวๆ เป็นบาท)
thai_salary_expectations = {
    'Entry Level': {
        'Min': 15000,
        'Max': 30000,
        'Thai': 'ผู้เริ่มต้น',
        'Position': ['จูเนียร์', 'เข้าใหม่', 'ผู้ช่วย']
    },
    'Junior': {
        'Min': 20000,
        'Max': 40000,
        'Thai': 'จูเนียร์',
        'Experience': '1-2 ปี'
    },
    'Mid Level': {
        'Min': 40000,
        'Max': 70000,
        'Thai': 'ระดับกลาง',
        'Experience': '3-5 ปี'
    },
    'Senior': {
        'Min': 60000,
        'Max': 120000,
        'Thai': 'อาวุโส',
        'Experience': '5-10 ปี'
    },
    'Lead/Manager': {
        'Min': 80000,
        'Max': 150000,
        'Thai': 'หัวหน้า/ผู้จัดการ',
        'Experience': '7+ ปี'
    },
    'Senior Manager': {
        'Min': 120000,
        'Max': 250000,
        'Thai': 'ผู้จัดการระดับสูง',
        'Experience': '10+ ปี'
    },
    'Executive': {
        'Min': 200000,
        'Max': float('inf'),
        'Thai': 'ผู้บริหารระดับสูง',
        'Experience': '15+ ปี',
        'Position': ['CEO', 'CTO', 'CFO', 'VP']
    }
}

# 📋 Dictionary สำหรับกระบวนการสัมภาษณ์ไทย
thai_interview_process = {
    'Phone Interview': ['สัมภาษณ์ทางโทรศัพท์', 'screening call', 'technical call'],
    'Video Interview': ['สัมภาษณ์ทางวิดีโอ', 'zoom interview', 'online interview'],
    'First Round': ['รอบแรก', 'HR interview', 'behavioral'],
    'Second Round': ['รอบที่สอง', 'technical interview', 'skills test'],
    'Third Round': ['รอบที่สาม', 'manager interview', 'management round'],
    'Final Round': ['รอบสุดท้าย', 'executive interview', 'final offer'],
    'Written Test': ['ข้อสอบเขียน', 'assessment', 'exam'],
    'Practical Test': ['ข้อสอบปฏิบัติ', 'coding test', 'case study'],
}

# 🎓 Dictionary สำหรับการศึกษาไทย
thai_education_system = {
    'High School': {
        'Thai': 'มัธยมปลาย',
        'Abbr': 'ม.6',
        'Years': '3 years',
        'Age': '15-18'
    },
    'Vocational': {
        'Thai': 'ปวช/ปวส',
        'Full': 'ปวิชนอุตสาหกรรม/ปวสอุตสาหกรรม',
        'Years': '2-3 years'
    },
    'Bachelor': {
        'Thai': 'ปริญญาตรี',
        'Abbr': 'ป.ตรี',
        'Years': '4 years',
        'Typical_GPA': '2.5 - 3.5'
    },
    'Master': {
        'Thai': 'ปริญญาโท',
        'Abbr': 'ป.โท',
        'Years': '2 years'
    },
    'PhD': {
        'Thai': 'ปริญญาเอก',
        'Abbr': 'ป.เอก',
        'Years': '3-5 years'
    }
}

# 🏛️ Dictionary สำหรับหน่วยงานภาครัฐ/ธุรกิจขนาดใหญ่ที่เป็นที่รู้จัก
thai_major_employers = {
    'Banking': ['ธนาคารกรุงไทย', 'ธนาคารกสิกรไทย', 'ธนาคารกรุงเทพ', 'ธนาคารทีเอมบี', 'Bangkok Bank'],
    'Automotive': ['ทีโอทา', 'อีซูซุ', 'ฮอนด้า', 'นิสสัน', 'BMW', 'Toyota'],
    'Energy': ['ปตท.', 'ประตะนะคม', ' MEA', 'PEA', 'Shell'],
    'Telecom': ['เอไอเอส', 'ดีแทค', 'ทรู', 'CAT Telecom', 'TOT'],
    'Retail': ['บิ๊กซี', 'เซ็นทรัล', 'คลับมอール', 'เอเอเอ็น', 'Tesco Lotus'],
    'Food & Beverage': ['บราววิ่ง', 'โอเตอร์สโตน', 'ลอตัส', 'ปตท.', 'Betagro'],
    'Technology': ['สุธัญญา', 'นอร์ทเทล', 'ไทยโทรเวอร์', 'เอพีสี', 'Google'],
    'Government': ['ราชการ', 'มหาวิทยาลัยแห่งชาติ', 'มหาวิทยาลัยเทคโนโลยีสิรินธร', 'องค์การสถาบันทดสอบสินค้าเกษตร'],
}

# ⏱️ Dictionary สำหรับวันหยุดของไทย (ส่งผลต่อการรับสมัคร)
thai_holidays = {
    'New Year Day': ['1 มกราคม'],
    'Makha Bucha': ['ขึ้นบูชาพระศาสน์สมัยศรีลังกา'],
    'Chakri Memorial Day': ['6 เมษายน'],
    'Songkran Festival': ['13-15 เมษายน'],
    'Labour Day': ['1 พฤษภาคม'],
    'Coronation Day': ['5 พฤษภาคม'],
    'Queen Suthida Birthday': ['3 มิถุนายน'],
    'Asahna Bucha': ['วันเนื่องจากพระเพ็ญเพ็ญ'],
    'Buddhist Lent': ['วันเข้าพรรษา'],
    'King Birthday': ['28 กรกฎาคม'],
    'Queen Mother Birthday': ['12 สิงหาคม'],
    'Constitution Day': ['17 ตุลาคม'],
    'King Bhumibol Memorial Day': ['13 ตุลาคม'],
    'Chulalongkorn Memorial Day': ['5 ธันวาคม'],
    'King Bhumibol Birthday': ['5 ธันวาคม'],
    'New Year Eve': ['31 ธันวาคม'],
}

# 🌐 Dictionary สำหรับภาษาอังกฤษในบริบทไทย
english_terms_in_thai_context = {
    'Manager': 'ผู้จัดการ',
    'Supervisor': 'ผู้บังคับบัญชา',
    'Staff': 'พนักงาน',
    'Trainee': 'คนฝึกงาน',
    'Temporary Staff': 'พนักงานชั่วคราว',
    'Permanent': 'พนักงานประจำ',
    'Part-time': 'พนักงานเสาร์อาทิตย์',
    'Full-time': 'พนักงานประจำเต็มเวลา',
    'Consultant': 'ที่ปรึกษา',
    'Executive': 'ผู้บริหาร',
}

print("✅ สร้างเสร็จ - Dictionary เฉพาะตลาดงานไทย")
print(f"   📌 thai_industries: {len(thai_industries)} อุตสาหกรรม")
print(f"   📌 thai_major_cities: {len(thai_major_cities)} เมืองหลัก")
print(f"   📌 thai_salary_expectations: {len(thai_salary_expectations)} ระดับเงินเดือน")
print(f"   📌 thai_interview_process: {len(thai_interview_process)} ขั้นตอน")
print(f"   📌 thai_education_system: {len(thai_education_system)} ระดับการศึกษา")
print(f"   📌 thai_major_employers: {len(thai_major_employers)} ประเภทบริษัท")
print(f"   📌 thai_holidays: {len(thai_holidays)} วันหยุด")
print(f"\n   🇹🇭 ข้อมูลตลาดงานไทยพร้อมใช้งาน!")

✅ สร้างเสร็จ - Dictionary เฉพาะตลาดงานไทย
   📌 thai_industries: 12 อุตสาหกรรม
   📌 thai_major_cities: 4 เมืองหลัก
   📌 thai_salary_expectations: 7 ระดับเงินเดือน
   📌 thai_interview_process: 8 ขั้นตอน
   📌 thai_education_system: 5 ระดับการศึกษา
   📌 thai_major_employers: 8 ประเภทบริษัท
   📌 thai_holidays: 16 วันหยุด

   🇹🇭 ข้อมูลตลาดงานไทยพร้อมใช้งาน!


In [31]:
# ---------------------------------------------------------
# 8. ชมสรุปของข้อมูลจากการจัดกลุ่ม (สองภาษา)
# ---------------------------------------------------------

# 🔴 ตรวจสอบว่า df มีคอลัมน์ที่จำเป็นหรือไม่
try:
    if 'Job_Category' not in df.columns:
        print("⚠️  DataFrame ยังไม่มีคอลัมน์ที่จัดกลุ่ม")
        print("   โปรดรัน Cell ที่ 7 ก่อน")
        import sys
        sys.exit()
except NameError:
    print("⚠️  ต้องรัน Cell ที่ 3 โหลด DataFrame ก่อน")
    import sys
    sys.exit()

if 'translate_to_thai' not in dir():
    print("⚠️  ต้องรัน Cell ที่มี Thai translation functions ก่อน")
    import sys
    sys.exit()

print("=" * 80)
print("📈 สรุปข้อมูลจากการจัดกลุ่ม / Data Summary")
print("=" * 80)

# สรุปโดยหมวดหมู่งาน
print("\n🏢 อันดับ TOP 5 ประเภทงาน / Top 5 Job Categories:")
print("-" * 80)
job_category_summary = df['Job_Category'].value_counts().head(5)
for i, (category, count) in enumerate(job_category_summary.items(), 1):
    percentage = (count / len(df)) * 100
    thai_category = translate_to_thai(category)
    print(f"   {i}. {category} ({thai_category}): {count:,} งาน ({percentage:.1f}%)")

# สรุปตามระดับประสบการณ์
print("\n👨‍💼 ความต้องการตามระดับประสบการณ์ / Experience Levels Required:")
print("-" * 80)
exp_summary = df['Experience_Level'].value_counts()
for level, count in exp_summary.items():
    percentage = (count / len(df)) * 100
    thai_level = translate_to_thai(level)
    print(f"   • {level} ({thai_level}): {count:,} ตำแหน่ง ({percentage:.1f}%)")

# สรุปตามประเภทบริษัท
print("\n🏭 อันดับ TOP 5 ประเภทบริษัท / Top 5 Company Types:")
print("-" * 80)
company_summary = df['Company_Type'].value_counts().head(5)
for i, (comp_type, count) in enumerate(company_summary.items(), 1):
    percentage = (count / len(df)) * 100
    print(f"   {i}. {comp_type}: {count:,} บริษัท ({percentage:.1f}%)")

# สรุปตามสถานที่
print("\n📍 อันดับ TOP 5 สถานที่ทำงาน / Top 5 Locations:")
print("-" * 80)
location_summary = df['Job_Location'].value_counts().head(5)
for i, (location, count) in enumerate(location_summary.items(), 1):
    percentage = (count / len(df)) * 100
    print(f"   {i}. {location}: {count:,} งาน ({percentage:.1f}%)")

# สรุปสวัสดิการ (Top Keywords)
print("\n🎁 สวัสดิการที่เจอบ่อยที่สุด / Most Common Benefits:")
print("-" * 80)
all_benefits = []
for benefits_str in df['Benefits']:
    if benefits_str and benefits_str != 'Not specified':
        all_benefits.extend([b.strip() for b in str(benefits_str).split(',')])
from collections import Counter
benefit_counts = Counter(all_benefits)
for benefit, count in benefit_counts.most_common(5):
    percentage = (count / len(df)) * 100
    thai_benefit = translate_to_thai(benefit)
    print(f"   • {benefit} ({thai_benefit}): {count:,} ตำแหน่ง ({percentage:.1f}%)")

print("\n" + "=" * 80)

📈 สรุปข้อมูลจากการจัดกลุ่ม / Data Summary

🏢 อันดับ TOP 5 ประเภทงาน / Top 5 Job Categories:
--------------------------------------------------------------------------------
   1. Development (พัฒนาซอฟต์แวร์): 908 งาน (36.3%)
   2. Other (Other): 673 งาน (26.9%)
   3. Business & Management (ธุรกิจและบริหารจัดการ): 409 งาน (16.4%)
   4. Data & Analytics (ข้อมูลและการวิเคราะห์): 262 งาน (10.5%)
   5. Quality & Testing (การควบคุมคุณภาพและทดสอบ): 80 งาน (3.2%)

👨‍💼 ความต้องการตามระดับประสบการณ์ / Experience Levels Required:
--------------------------------------------------------------------------------
   • Entry Level (ผู้เริ่มต้น): 1,032 ตำแหน่ง (41.3%)
   • Not specified (Not specified): 503 ตำแหน่ง (20.1%)
   • Senior Level (ระดับสูง): 448 ตำแหน่ง (17.9%)
   • Mid Level (ระดับกลาง): 421 ตำแหน่ง (16.8%)
   • Management (บริหารจัดการ): 95 ตำแหน่ง (3.8%)

🏭 อันดับ TOP 5 ประเภทบริษัท / Top 5 Company Types:
--------------------------------------------------------------------------------
   

In [32]:


# ---------------------------------------------------------
# 9. บันทึกข้อมูลที่จัดกลุ่มแล้ว (CSV + JSON)
# ---------------------------------------------------------

# 🔴 ตรวจสอบความพร้อม
try:
    if 'Job_Category' not in df.columns:
        print("⚠️  DataFrame ยังไม่มีคอลัมน์ที่จัดกลุ่ม")
        print("   โปรดรัน Cell ที่ 7 (categorization) ก่อน")
        import sys
        sys.exit()
except NameError:
    print("⚠️  DataFrame ยังไม่ได้สร้าง")
    print("   โปรดรัน Cell ที่ 3 (Create DataFrame) ก่อน")
    import sys
    sys.exit()

# เตรียมข้อมูลสำหรับบันทึก (ลบ Full_Description ทำให้ไฟล์มีขนาดเล็ก)
df_final = df.drop(columns=['Full_Description']) if 'Full_Description' in df.columns else df.copy()

# บันทึกไฟล์ CSV ที่มีการจัดกลุ่มแล้ว
csv_enriched = 'enriched_jobs_with_categorization.csv'
df_final.to_csv(csv_enriched, index=False, encoding='utf-8-sig')
print(f"✅ บันทึกไฟล์ '{csv_enriched}' เสร็จสิ้น")

# บันทึกไฟล์ JSON ที่มีการจัดกลุ่มแล้ว
json_enriched = 'enriched_jobs_with_categorization.json'
df_final.to_json(json_enriched, orient='records', force_ascii=False, indent=2)
print(f"✅ บันทึกไฟล์ '{json_enriched}' เสร็จสิ้น")

# แสดงป้ายกำกับคอลัมน์ที่มีให้ใช้
print("\n📋 คอลัมน์ที่เก็บไว้ในไฟล์:")
thai_mapping_simple = {
    'Job_Category': 'หมวดหมู่งาน',
    'Experience_Level': 'ระดับประสบการณ์',
    'Company_Type': 'ประเภทบริษัท',
    'Job_Location': 'สถานที่ทำงาน',
    'Benefits': 'สวัสดิการ',
    'Skills': 'ทักษะที่ต้องการ',
}
for i, col in enumerate(df_final.columns, 1):
    thai_label = thai_mapping_simple.get(col, col)
    print(f"   {i:2d}. {col} ({thai_label})")

print(f"\n✨ สิ้นสุดกระบวนการประมวลผล!")
print(f"   📊 ได้ข้อมูล {len(df_final):,} งาน พร้อมใช้งาน")
print(f"   📁 ไฟล์ที่บันทึก: CSV + JSON")
print(f"   🎉 ไฟล์มีขนาดเล็กลง เพราะไม่มีข้อมูลซ้ำซ้อน")



✅ บันทึกไฟล์ 'enriched_jobs_with_categorization.csv' เสร็จสิ้น
✅ บันทึกไฟล์ 'enriched_jobs_with_categorization.json' เสร็จสิ้น

📋 คอลัมน์ที่เก็บไว้ในไฟล์:
    1. Keyword (Keyword)
    2. Job_Title (Job_Title)
    3. Company (Company)
    4. Job_Description (Job_Description)
    5. Job_Link (Job_Link)
    6. Source_Page (Source_Page)
    7. Skills (ทักษะที่ต้องการ)
    8. Job_Category (หมวดหมู่งาน)
    9. Benefits (สวัสดิการ)
   10. Job_Location (สถานที่ทำงาน)
   11. Experience_Level (ระดับประสบการณ์)
   12. Company_Type (ประเภทบริษัท)
   13. Job_Type (Job_Type)
   14. Education_Required (Education_Required)
   15. Languages (Languages)
   16. Soft_Skills (Soft_Skills)
   17. Department (Department)
   18. Work_Arrangement (Work_Arrangement)
   19. Primary_Function (Primary_Function)
   20. Tech_Stack (Tech_Stack)

✨ สิ้นสุดกระบวนการประมวลผล!
   📊 ได้ข้อมูล 2,499 งาน พร้อมใช้งาน
   📁 ไฟล์ที่บันทึก: CSV + JSON
   🎉 ไฟล์มีขนาดเล็กลง เพราะไม่มีข้อมูลซ้ำซ้อน


In [33]:

# ---------------------------------------------------------
# 10. ชมตัวอย่างของข้อมูลที่จัดกลุ่มแล้ว
# ---------------------------------------------------------

try:
    if 'df_final' not in dir():
        print("⚠️  ต้องรัน Cell ที่ 9 (บันทึกข้อมูล) ก่อน")
        import sys
        sys.exit()
except NameError:
    print("⚠️  DataFrame ยังไม่พร้อม")
    import sys
    sys.exit()

print("📋 ตัวอย่างข้อมูลที่จัดกลุ่มแล้ว (5 แถวแรก):\n")
showcase_columns = ['Job_Title', 'Company', 'Job_Category', 'Experience_Level', 
                     'Company_Type', 'Job_Location', 'Skills']
display(df_final[showcase_columns].head())


📋 ตัวอย่างข้อมูลที่จัดกลุ่มแล้ว (5 แถวแรก):



📋 ตัวอย่างข้อมูลที่จัดกลุ่มแล้ว (5 แถวแรก):



,Job_Title,Company,Job_Category,Experience_Level,Company_Type,Job_Location,Skills
0,"Full Stack Developer (C#, .Net Core, React, Vu...","CHOCO CARD ENTERPRISE CO., LTD.",Development,Not specified,Other,Bangkok,"java, c#, javascript, go, react, html, css, ex..."
1,(Junior Level) Backend Developer,Bangkok Bank Public Company Limited,Development,Not specified,Financial,Bangkok,"java, c#, c++, go, sql, oracle, agile, scrum, ..."
2,Site Reliability Engineer (SRE) – Cloud & DevOps,Cathcart Associates Asia Recruitment Ltd.,Development,Senior Level,Technology,Bangkok,"aws, azure, gcp, docker, kubernetes, jenkins, ..."
3,Master Data Management Officer,THAI UNION GROUP PCL.,Database Administration,Mid Level,Other,Northern,"go, sap"
4,Data Engineer,TPCS Public Company Limited,Development,Not specified,Technology,Not specified,"python, java, go, sql, aws, azure, gcp, agile,..."


In [34]:

# ---------------------------------------------------------
# 11. ชมตัวอย่างข้อมูลขั้นสุดท้าย
# ---------------------------------------------------------

try:
    if 'df_final' not in dir():
        print("⚠️  ต้องรัน Cell ที่ 9 (บันทึกข้อมูล) ก่อน")
        import sys
        sys.exit()
except NameError:
    print("⚠️  DataFrame ยังไม่พร้อม")
    import sys
    sys.exit()

print("📋 ตัวอย่างข้อมูลที่จัดกลุ่มแล้ว (5 แถวแรก):\n")
showcase_columns = ['Job_Title', 'Company', 'Job_Category', 'Experience_Level', 
                     'Company_Type', 'Job_Location', 'Skills']
display(df_final[showcase_columns].head())

print("\n✨ ประมวลผลเสร็จสิ้น - ไฟล์พร้อมใช้งาน!")


📋 ตัวอย่างข้อมูลที่จัดกลุ่มแล้ว (5 แถวแรก):



📋 ตัวอย่างข้อมูลที่จัดกลุ่มแล้ว (5 แถวแรก):



,Job_Title,Company,Job_Category,Experience_Level,Company_Type,Job_Location,Skills
0,"Full Stack Developer (C#, .Net Core, React, Vu...","CHOCO CARD ENTERPRISE CO., LTD.",Development,Not specified,Other,Bangkok,"java, c#, javascript, go, react, html, css, ex..."
1,(Junior Level) Backend Developer,Bangkok Bank Public Company Limited,Development,Not specified,Financial,Bangkok,"java, c#, c++, go, sql, oracle, agile, scrum, ..."
2,Site Reliability Engineer (SRE) – Cloud & DevOps,Cathcart Associates Asia Recruitment Ltd.,Development,Senior Level,Technology,Bangkok,"aws, azure, gcp, docker, kubernetes, jenkins, ..."
3,Master Data Management Officer,THAI UNION GROUP PCL.,Database Administration,Mid Level,Other,Northern,"go, sap"
4,Data Engineer,TPCS Public Company Limited,Development,Not specified,Technology,Not specified,"python, java, go, sql, aws, azure, gcp, agile,..."



✨ ประมวลผลเสร็จสิ้น - ไฟล์พร้อมใช้งาน!
